In [1]:
from langchain_ollama import OllamaEmbeddings
# 创建向量模型,我们今天使用ollama

ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

# 初始化向量数据库客户端对象
from pymilvus import MilvusClient
from app.core.config import settings

# 初始化客户端
milvus_client = MilvusClient(uri=settings.rag.milvus_url)

# Collection名称:集合,指的就是表名字
collection_name = "my_collection_1"

In [2]:
# 定义向量检索函数
def dense_query(question: str) -> list[list['dict']]:
    """
    输入用户问题匹配向量
    :param question: 用户问题
    :return: 返回的结果是一个列表,返回多条向量匹配的结果,每一个子列表是匹配的文本片段
    """
    # 将用户问题向量化
    user_question = ollama_embeddings.embed_query(question)

    # 设置查询参数
    results = milvus_client.search(
        # 集合名
        collection_name=collection_name,
        # 根据哪一个字段做匹配
        anns_field='dense',
        # 输入用户问题的向量
        data=[user_question],
        # 最多匹配三条
        limit=3,
        search_params={
            # 使用余弦算法匹配
            "metric_type":"COSINE"
        },
        # 返回三个字段的结果
        output_fields=['id','h2','content']
    )
    return results

In [3]:
import json
# 调用函数遍历结果打印
results = dense_query('教育的负面结果是什么')

# 循环遍历
for hits in results:
    for hit in  hits:
        print(json.dumps(hit,indent=2,ensure_ascii=False))

{
  "id": 15,
  "distance": 0.5672498941421509,
  "entity": {
    "h2": "第一节 教育的功能",
    "content": "# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。",
    "id": 15
  }
}
{
  "id": 16,
  "distance": 0.5511353015899658,
  "entity": {
    "h2": "第二节 教育和社会的关系",
    "content": "## 第二节 教育和社会的关系  \n### （一）政治与教育的关系  \n政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  \n教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。",
    "id": 16
  }
}
{
  "id": 9,
  "distance": 0.5489168167114258,
  "entity": {
    "h2": "第二节 教育的定义",
    "content": "## 第二节 教育的定义  \n### （一）教育的定义  \n广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  \n狭义的教育就是指学校教育。  \n教育的要素：教育者、受教育者、教育影响（主要是教育内容）。",
    "id": 9
  }
}


In [4]:
# 关键词匹配结果
results = dense_query('教育漫画是谁写的')

# 循环遍历
for hits in results:
    for hit in  hits:
        print(json.dumps(hit,indent=2,ensure_ascii=False))

{
  "id": 4,
  "distance": 0.19160450994968414,
  "entity": {
    "h2": "第一节 中外教育家及其教育思想",
    "content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。",
    "id": 4
  }
}
{
  "id": 6,
  "distance": 0.19037774205207825,
  "entity": {
    "h2": "第一节 中外教育家及其教育思想",
    "content": "### （六）希腊三贤  \n**苏格拉底**：产婆术（谈话法），分三步：讽刺—定义—助产术。国外启发式教育第一人。  \n**柏拉图**：《理想国》提出普及教育的主张，教育目的是培养哲学王。  \n**亚里士多德**：提出自由教育，提倡对学生进行和谐全面发展的教育，灵魂说。",
    "id": 6
  }
}
{
  "id": 8,
  "distance": 0.17329196631908417,
  "entity": {
    "h2": "第一节 中外教育家及其教育思想",
    "content": "### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解